# Personal Hiking Time Predictor: prepare and explore your data

This notebook introduces **exploratory data analysis (EDA)**: loading, checking, summarizing, and plotting recorded GPX data before choosing a model. It does not train a regression model.

First run the importer described in the README, then run these cells from top to bottom. The main analysis uses your real recordings and configurable preprocessing settings. The original manual CSV exploration remains available afterward. No sample hikes are generated.

## Understand features and the future target

In the original manual CSV, each row represents **one completed hike**. The GPX importer instead creates rows for approximately half-mile segments. A segment is a smaller piece of a hike; neighboring segments are related, so they are not independent hikes. Total historical miles are not a count of independent training examples.

| Column | Meaning | Role in later modeling |
| --- | --- | --- |
| `hike_id` | Unique, nonblank identifier | Identifier, excluded from model inputs |
| `distance_miles` | Distance traveled, greater than 0 | Numerical feature |
| `elevation_gain_ft` | Total ascent, 0 or greater | Numerical feature |
| `elevation_loss_ft` | Total descent as a positive magnitude, 0 or greater | Numerical feature |
| `hike_type` | `day_hike` or `backpacking`; missing until supplied for GPX | Categorical feature |
| `moving_time_min` | Time spent moving in minutes, greater than 0 | Target |

A **feature** is an input used for prediction. The **target** is the value we eventually want to predict. Because moving time is a continuous numerical quantity, this is a **regression** problem. Hike type will need categorical encoding later.

## Import tools and locate the project

First follow the README setup commands and select the virtual environment's Python kernel. Start Jupyter from the project root or its `notebooks/` folder.

pandas handles tables, and Matplotlib draws charts. The reusable loader in `src/hiking_time_predictor/data.py` uses pandas and NumPy to validate measurements. scikit-learn is installed for a later stage.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from hiking_time_predictor.data import HIKE_TYPES, NUMERIC_COLUMNS, load_hiking_data

pd.set_option('display.max_columns', None)

# Support launching Jupyter from either the root or notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

TEMPLATE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hikes_template.csv'
if not TEMPLATE_PATH.is_file():
    raise FileNotFoundError('Start Jupyter from the project root or notebooks folder.')

print(f'Project folder: {PROJECT_ROOT}')

## Load the GPX preparation results

The importer reads GPX **trackpoints**, not waypoints or routes. `hiking_summary.csv` has one row per recording; `hiking_segments.csv` has one row per distance segment. Trackpoint and interval tables let us check how each number was calculated. Miles, feet, and minutes are named explicitly in the columns.

Run `python -m hiking_time_predictor.import_gpx --config config/gpx_import.json` from the project root whenever recordings, metadata, or thresholds change. This notebook only reads the results, so it never changes an original recording.

In [ ]:
import json
import numpy as np
from hiking_time_predictor.data import select_model_ready_segments

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
if not (PROCESSED / 'hiking_segments.csv').exists():
    raise FileNotFoundError('Run python -m hiking_time_predictor.import_gpx --config config/gpx_import.json first.')

segments = pd.read_csv(PROCESSED / 'hiking_segments.csv', dtype={'hike_id': 'string', 'hike_type': 'string'})
summary = pd.read_csv(PROCESSED / 'hiking_summary.csv', dtype={'hike_id': 'string', 'hike_type': 'string'})
trackpoints = pd.read_csv(PROCESSED / 'hiking_trackpoints.csv')
intervals = pd.read_csv(PROCESSED / 'hiking_intervals.csv')
settings = json.loads((PROCESSED / 'import_settings.json').read_text())
for table, columns in [(segments, ['start_time_utc', 'end_time_utc']),
                       (summary, ['start_time_utc', 'end_time_utc']),
                       (trackpoints, ['time_utc']),
                       (intervals, ['start_time_utc', 'end_time_utc'])]:
    for column in columns:
        table[column] = pd.to_datetime(table[column], utc=True, format='mixed')
for table in [segments, summary, trackpoints, intervals]:
    table['quality_flags'] = table['quality_flags'].fillna('')

print(f'{len(summary)} recorded hikes; {len(segments)} distance segments')
display(summary[['hike_id', 'source_file', 'trackpoint_count', 'distance_miles',
                 'elapsed_time_min', 'moving_time_min', 'stationary_time_min', 'unknown_time_min']])
print('Settings used for this import:')
display(pd.Series(settings['config'], name='value'))

## Choose a recording to inspect

Change `selected_hike_id` below to any ID in the summary table. We start with Morning hike when available. **Elapsed time** is the time between the first and last known timestamps. **Moving time** is an estimate using GPS displacement and timing. **Stationary time** is likely rest or GPS drift. **Unknown time** is an interval we cannot classify reliably.

By default, gaps longer than 120 seconds remain unknown. Short intervals with speed at most 0.3 mph and displacement at most 10 meters are stationary. A sustained cluster within a 10-meter radius for at least 60 seconds also suggests a stop. Speeds above 8 mph are flagged as implausible. All these settings can be changed; slow walking and GPS drift can still confuse the heuristic.

**Hiker confirmations:** explicitly reviewed intervals in `data/confirmed_rests.csv` count as stationary time. Automatic detection still applies to unreviewed recordings. Confirmations never reconstruct missing GPS geometry.


In [ ]:
selected_hike_id = 'morning-hike' if 'morning-hike' in summary['hike_id'].values else summary.iloc[0]['hike_id']
chosen = summary.loc[summary['hike_id'].eq(selected_hike_id)].iloc[0]
pts = trackpoints.loc[trackpoints['hike_id'].eq(selected_hike_id)].copy()
parts = segments.loc[segments['hike_id'].eq(selected_hike_id)].copy()
steps = intervals.loc[intervals['hike_id'].eq(selected_hike_id)].copy()
pts['elapsed_min'] = (pts['time_utc'] - chosen['start_time_utc']).dt.total_seconds() / 60
print(f"Inspecting {chosen['source_file']}")
display(chosen[['raw_distance_miles', 'distance_miles', 'raw_elevation_gain_ft',
                'unsmoothed_moving_elevation_gain_ft', 'elevation_gain_ft', 'raw_elevation_loss_ft',
                'elevation_loss_ft', 'elapsed_time_min', 'moving_time_min',
                'stationary_time_min', 'unknown_time_min', 'detected_unknown_time_min',
                'confirmed_rest_count', 'confirmed_rest_time_min', 'quality_flags']])
if pd.notna(chosen['elapsed_time_min']):
    classified = chosen[['moving_time_min', 'stationary_time_min', 'unknown_time_min']].sum()
    assert np.isclose(classified, chosen['elapsed_time_min'])
    assert np.isclose(parts['elapsed_time_min'].sum(), chosen['elapsed_time_min'])

## 1. Raw and cleaned elevation profile

The horizontal axis is accepted distance; the vertical axis is altitude in feet. Gray dots show original readings. The blue curve uses a **rolling median**: the middle elevation in a local distance window. First we resample to a 5-meter distance grid; then use a centered 30-meter window. Equal-distance readings at a stop are reduced to their median so dense stationary samples do not dominate the filter.

Noise creates many small artificial climbs and descents. Smoothing removes some of them, but can also flatten real short climbs and endpoints. We never filter across unknown intervals, missing measurements, or GPX boundaries; separated curves are intentional. Cleaned gain totals include only accepted moving intervals. The intermediate unsmoothed-moving total below separates interval exclusions from the effect of smoothing. These are estimates for observed sections, not recovered elevation across gaps.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(pts['distance_miles'], pts['elevation_ft'], s=5, color='gray', alpha=0.5, label='Raw GPS elevation')
first = True
for block_id, block in pts.loc[pts['profile_block'].ge(0)].groupby('profile_block', sort=False):
    ax.plot(block['distance_miles'], block['smoothed_elevation_ft'], color='tab:blue',
            label='Smoothed elevation' if first else None)
    first = False
ax.set(xlabel='Accepted distance (miles)', ylabel='Elevation (feet)', title=f'Elevation: {selected_hike_id}')
ax.legend(); ax.grid(alpha=0.25); fig.tight_layout(); plt.show()
display(pd.Series({
    'Raw ascent, all adjacent recorded points (ft)': chosen['raw_elevation_gain_ft'],
    'Raw ascent, accepted moving intervals only (ft)': chosen['unsmoothed_moving_elevation_gain_ft'],
    'Smoothed ascent, accepted moving intervals only (ft)': chosen['elevation_gain_ft'],
}, name='ascent_ft'))

## 2. GPS distance versus elapsed time

Each dot is an actual GPS reading. A steep increase means distance accumulated quickly; a flat region suggests a stop or unclassified time. Green shading marks intervals you confirmed as rests. Orange shading marks any remaining unknown intervals. For future unreviewed recordings, **a gap with nearby endpoints could be a rest or a missing recording**; the flat accepted-distance curve alone does not prove a rest.

Gray dots accumulate raw point-to-point distance, including GPS drift and straight chords between gap endpoints. Blue dots accumulate only accepted moving distance. We draw dots instead of connecting through gaps: neither path nor travel speed is known inside them.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(pts['elapsed_min'], pts['raw_distance_miles'], s=5, color='gray', alpha=0.5, label='Raw cumulative GPS distance')
ax.scatter(pts['elapsed_min'], pts['distance_miles'], s=5, color='tab:blue', label='Accepted moving distance')
review_intervals = steps.loc[steps['state'].eq('unknown') | steps['state_source'].eq('user_confirmed')]
labels_drawn = set()
for step in review_intervals.itertuples():
    start = (step.start_time_utc - chosen['start_time_utc']).total_seconds() / 60
    end = (step.end_time_utc - chosen['start_time_utc']).total_seconds() / 60
    confirmed = step.state_source == 'user_confirmed'
    label = 'Confirmed rest' if confirmed else 'Unknown interval'
    color = 'tab:green' if confirmed else 'tab:orange'
    ax.axvspan(start, end, color=color, alpha=0.18, label=label if label not in labels_drawn else None)
    labels_drawn.add(label)
ax.set(xlabel='Elapsed time (minutes)', ylabel='Distance (miles)', title='GPS distance and time')
ax.legend(); ax.grid(alpha=0.25); fig.tight_layout(); plt.show()
print('Confirmed rests and any remaining unknown intervals:')
display(review_intervals[['start_time_utc', 'end_time_utc', 'elapsed_seconds', 'endpoint_distance_m',
                          'detected_state', 'state', 'state_source', 'review_notes', 'quality_flags']])

## 3. Segment moving pace

**Pace** is minutes per mile: moving minutes divided by accepted miles. A taller bar means a slower segment. Stops and unknown time are not counted as moving minutes. The last segment can be shorter; pace divides by its actual distance. Zero-distance or zero-moving-time rows have no useful pace.

Orange bars identify measurement flags, such as a recording gap. Missing hike type is shown in the table but does not change this numeric graph. Flagged pace values describe only accepted intervals and need review before modeling.

In [ ]:
positive = parts['distance_miles'].gt(0) & parts['moving_time_min'].gt(0)
parts['moving_pace_min_per_mile'] = (parts['moving_time_min'] / parts['distance_miles'].replace(0, np.nan)).where(positive)
parts['moving_speed_mph'] = (parts['distance_miles'] / (parts['moving_time_min'].replace(0, np.nan) / 60)).where(positive)
parts['measurement_flagged'] = parts['quality_flags'].map(lambda text: bool(set(text.split(';')) - {'', 'missing_hike_type'}))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(parts['segment_index'], parts['moving_pace_min_per_mile'],
       color=np.where(parts['measurement_flagged'], 'tab:orange', 'tab:blue'))
ax.set(xlabel='Segment index (starts at 0)', ylabel='Moving pace (minutes per mile)', title='Estimated segment moving pace')
ax.grid(axis='y', alpha=0.25); fig.tight_layout(); plt.show()

## 4. Segment elevation gain versus moving time

Each dot is one segment: cleaned uphill gain on the horizontal axis, estimated moving minutes on the vertical axis. An upward pattern may suggest climbing takes more time. A short final segment has less distance, so comparisons are imperfect. Terrain, fatigue, downhill travel, and GPS uncertainty also matter. A scatter plot shows association, not causation or model accuracy.

Orange dots have measurement flags. We keep them visible for learning and investigation; a future training selection excludes flagged rows.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for flagged, color, label in [(False, 'tab:blue', 'No measurement flags'), (True, 'tab:orange', 'Measurement flags')]:
    group = parts.loc[parts['measurement_flagged'].eq(flagged)]
    if not group.empty:
        ax.scatter(group['elevation_gain_ft'], group['moving_time_min'], color=color, label=label)
ax.set(xlabel='Cleaned elevation gain (feet)', ylabel='Estimated moving time (minutes)', title='Ascent and segment moving time')
ax.legend(); ax.grid(alpha=0.25); fig.tight_layout(); plt.show()

## 5. Distribution of segment hiking speeds

**Speed** is miles per hour: accepted miles divided by moving hours. Unlike pace, higher speed means faster travel. This histogram groups speeds into ranges; bar height counts segments in each range. With just a few segments, the distribution is unstable and cannot establish a typical speed for all future hikes.

This chart includes segments with measurement flags for EDA. It describes the heuristic's estimates, not verified speeds. Try changing thresholds in the config, reimporting, and rerunning the notebook to see how sensitive the results are.

In [ ]:
speeds = parts['moving_speed_mph'].dropna()
fig, ax = plt.subplots(figsize=(7, 4))
if not speeds.empty:
    ax.hist(speeds, bins=min(6, len(speeds)), color='tab:blue', edgecolor='white')
else:
    print('No measured moving segments for this recording.')
ax.set(xlabel='Estimated moving speed (miles per hour)', ylabel='Number of segments', title='Segment speed distribution')
ax.grid(axis='y', alpha=0.25); fig.tight_layout(); plt.show()

## 6. Segment table and quality flags

A flag explains a limitation, not necessarily an error. `recording_gap` means timing was too sparse; `possible_rest_during_gap` means the automatic detector found nearby endpoints. `user_confirmed_rest` means you supplied the missing context: the hiker was stationary. That time counts as a rest, while the original GPS quality flags remain visible. `missing_hike_type` means you have not supplied a category. Other flags are defined in the README.

Each row contains moving, stationary, and unknown minutes; these add up to elapsed minutes. `stopped_or_unknown_time_min` combines the latter two, without claiming unknown time was spent resting. Boundaries inside valid consecutive moving intervals are interpolated under a constant-speed assumption. A bin can span a gap, but the gap itself is assigned intact and contributes no invented distance or elevation.

In [ ]:
display(parts[['hike_id', 'segment_index', 'distance_miles', 'elevation_gain_ft', 'elevation_loss_ft',
               'elapsed_time_min', 'moving_time_min', 'stationary_time_min', 'unknown_time_min',
               'stopped_or_unknown_time_min', 'start_time_utc', 'end_time_utc', 'hike_type', 'quality_flags']])
assert np.allclose(parts['elapsed_time_min'], parts['moving_time_min'] + parts['stationary_time_min'] + parts['unknown_time_min'])
print('Missing metadata by column:')
display(summary[['hike_type', 'pack_weight_lbs']].isna().sum())
ready = select_model_ready_segments(segments)
print(f'{len(ready)} complete, unflagged segments are eligible for a future model requiring hike type.')
print(f'{len(select_model_ready_segments(segments, require_pack_weight=True))} also have a known pack weight.')
print('No model is trained. Edit data/hike_metadata.csv only with information you know, then reimport.')

## Reflect before modeling GPX segments

1. Check the saved rest confirmations against your memory of these hikes. For future recordings, review any remaining unknown intervals before assigning rest labels.
2. How much ascent changes after excluding uncertain intervals, and how much changes because of the median filter?
3. Do slower segments also have more climbing, or might another explanation fit?
4. Are your results sensitive to the smoothing window and movement thresholds?
5. Which metadata can you supply from your own records?

A future train/test split must keep entire **hikes** together. Randomly putting adjacent segments from the same recording into both sets can make a model look more accurate than it is. Historical miles are useful history, but the number of independent recordings and their variety still limit what you can learn.

## Optional: the original manual hike CSV exploration

The following cells retain the original six-column loader and graphs. They use `data/raw/hikes.csv` if you created it, otherwise the empty header-only template. GPX segments have a richer schema and must not be passed into that strict manual-hike loader.

## 3. Load and validate the CSV

Copy `data/raw/hikes_template.csv` to `data/raw/hikes.csv` once, then fill in only your actual observations. Keep all six headers. Use miles, feet, and minutes consistently, and record moving time rather than elapsed time including long breaks.

The loader checks required columns, missing values, unique IDs, hike categories, and finite numbers in valid ranges. If validation raises a `ValueError`, read its message, correct the CSV, and rerun this cell. An empty table with the right headers is valid at this stage.

In [ ]:
PERSONAL_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hikes.csv'
DATA_PATH = PERSONAL_DATA_PATH if PERSONAL_DATA_PATH.is_file() else TEMPLATE_PATH

hikes = load_hiking_data(DATA_PATH)
print(f'Loaded {len(hikes)} hikes from {DATA_PATH.name}.')
display(hikes.head())

if hikes.empty:
    print('No observations yet. Add your real hikes to data/raw/hikes.csv, then rerun all cells.')

## 4. Inspect shape, types, and missing values

`shape` gives the number of rows and columns. `info()` shows the data type and nonmissing count for each column. `isna().sum()` counts missing values. Validation already requires complete observations; these inspection tools help you understand what a DataFrame contains.

The following inspection does not alter the dataset.

In [ ]:
print(f'Rows: {hikes.shape[0]}, columns: {hikes.shape[1]}')
hikes.info()
display(hikes.isna().sum().rename('missing_values'))

if not hikes.empty:
    # Count each category, including a zero if that category is absent.
    hike_counts = hikes['hike_type'].value_counts().reindex(HIKE_TYPES, fill_value=0)
    display(hike_counts.rename('hike_count'))

## 5. Descriptive statistics

`describe()` summarizes the numerical columns:

- **count**: number of recorded values.
- **mean**: arithmetic average.
- **std**: sample standard deviation, a measure of spread; it is undefined (`NaN`) with only one observation.
- **min / max**: smallest and largest values.
- **25% / 50% / 75%**: quartiles; 50% is the median.

Compare units carefully: distance is in miles, elevation is in feet, and time is in minutes. Averages grouped by hike type describe the trips you recorded; differences may also reflect trip distance and elevation.

In [ ]:
if hikes.empty:
    print('Descriptive statistics will appear after you add real observations.')
else:
    display(hikes[NUMERIC_COLUMNS].describe())
    print('Mean measurements by hike type:')
    display(hikes.groupby('hike_type')[NUMERIC_COLUMNS].mean())

## 6. Plot features against moving time

Each point in a scatter plot is one recorded hike. The horizontal axis is a feature, and the vertical axis is the future target. Colors and marker shapes distinguish day hikes from backpacking trips.

Look for a relationship between distance and time, variation among hikes of similar distance, and possible effects of ascent or descent. An unusual point is a reason to check the source record; it is not automatically a mistake. These charts show associations, not causes or evidence of model accuracy.

In [ ]:
if hikes.empty:
    print('Scatter plots will appear after you add real observations.')
else:
    plot_features = [
        ('distance_miles', 'Distance (miles)', 'Distance and moving time'),
        ('elevation_gain_ft', 'Elevation gain (ft)', 'Ascent and moving time'),
        ('elevation_loss_ft', 'Elevation loss (ft)', 'Descent and moving time'),
    ]
    colors = {'day_hike': 'tab:blue', 'backpacking': 'tab:orange'}
    markers = {'day_hike': 'o', 'backpacking': '^'}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    for axis, (feature, x_label, title) in zip(axes, plot_features):
        for hike_type in HIKE_TYPES:
            group = hikes[hikes['hike_type'] == hike_type]
            if group.empty:
                continue
            axis.scatter(
                group[feature], group['moving_time_min'],
                label=hike_type, color=colors[hike_type],
                marker=markers[hike_type], alpha=0.8,
            )
        axis.set_xlabel(x_label)
        axis.set_ylabel('Moving time (minutes)')
        axis.set_title(title)
        axis.grid(alpha=0.25)
        axis.legend(title='Hike type')

    fig.tight_layout()
    plt.show()

## 7. Reflect before modeling

After you enter real observations, write down:

1. How many hikes do you have, and how many are in each category?
2. What range of distances and elevation changes does your history cover?
3. Which feature appears most associated with moving time?
4. Are any records inconsistent with their original source or units?
5. Which future hikes would fall outside the conditions you have recorded?

A small personal dataset can be useful for learning, but a few points cannot establish reliable prediction performance. In a later stage, we will encode hike type, define a train/test approach, fit a regression model, and evaluate prediction errors. For now, focus on accurate records and understanding the table.